In [7]:
from pathlib import Path
import pandas as pd

src_dir = Path("/projects/immunestatus/vdjrearm/sample")
dst_dir = Path(".")

exclude = {"TRA", "TRB"}
targets = {"IGH", "IGK", "IGL", "TRD", "TRG"}
MIN_SEQ_LEN = 7
MAX_SEQ_LEN = 40
def add_allele(gene):
    gene = str(gene).strip()
    if not gene:
        return gene
    if gene == "IGHV3-43D":
        return "IGHV3-43D*03"
    if "*" not in gene:
        return f"{gene}*01"
    return gene


def normalize_chain_df(df, chain, add_alleles=False):
    rename_map = {}
    if "junction" in df.columns and "junction_aa" not in df.columns:
        raise ValueError(f"{chain}: found 'junction' but no 'junction_aa'. Amino acid column is required.")
    if "cdr3aa" in df.columns and "junction_aa" not in df.columns:
        rename_map["cdr3aa"] = "junction_aa"
    if "v" in df.columns and "v_call" not in df.columns:
        rename_map["v"] = "v_call"
    if "j" in df.columns and "j_call" not in df.columns:
        rename_map["j"] = "j_call"

    df = df.rename(columns=rename_map)

    required = ["junction_aa", "v_call", "j_call"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{chain}: missing columns {missing}. Have: {list(df.columns)}")

    df = df[required].copy()
    df = df.dropna(subset=required)

    for col in required:
        df[col] = df[col].astype(str).str.strip()

    seq_len = df["junction_aa"].str.len()
    df = df.loc[seq_len.between(MIN_SEQ_LEN, MAX_SEQ_LEN)].copy()

    if chain == "TRD":
        df["v_call"] = df["v_call"].replace({
            "TRAV29DV5": "TRAV29/DV5*01",
            "TRAV29DV5*01": "TRAV29/DV5*01",
        })

    if chain == "IGK":
        mask = ~(
            df["v_call"].str.contains("IGKV1-NL1", na=False) |
            df["j_call"].str.contains("IGKV1-NL1", na=False)
        )
        df = df.loc[mask].copy()

    if add_alleles:
        df["v_call"] = df["v_call"].map(add_allele)
        df["j_call"] = df["j_call"].map(add_allele)
    elif chain == "IGH":
        df["v_call"] = df["v_call"].replace({
            "IGHV3-43D": "IGHV3-43D*03",
        })

    df = df.drop_duplicates(subset=required, keep="first")
    return df


def read_unique_rows(path, chain, target_n, skiprows=None, chunksize=50000, add_alleles=False, seen_keys=None):
    parts = []
    seen = set() if seen_keys is None else set(seen_keys)
    total = 0

    reader = pd.read_csv(
        path,
        sep="\t",
        compression="gzip",
        skiprows=skiprows,
        chunksize=chunksize,
    )

    for chunk in reader:
        chunk = normalize_chain_df(chunk, chain, add_alleles=add_alleles)
        if chunk.empty:
            continue

        keep_idx = []
        for idx, row in chunk.iterrows():
            key = (row["junction_aa"], row["v_call"], row["j_call"])
            if key in seen:
                continue
            seen.add(key)
            keep_idx.append(idx)
            total += 1
            if total >= target_n:
                break

        if keep_idx:
            parts.append(chunk.loc[keep_idx])

        if total >= target_n:
            break

    if not parts:
        return pd.DataFrame(columns=["junction_aa", "v_call", "j_call"])

    result = pd.concat(parts, ignore_index=True)
    if len(result) < target_n:
        print(f"Warning: {path.name} -> only {len(result)} unique rows after filtering/dedup")
    return result.head(target_n).copy()


def read_unique_rows_with_topup(path, chain, target_n, primary_skiprows=None, chunksize=50000, add_alleles=False):
    result = read_unique_rows(
        path,
        chain,
        target_n=target_n,
        skiprows=primary_skiprows,
        chunksize=chunksize,
        add_alleles=add_alleles,
    )

    if len(result) >= target_n:
        return result.head(target_n).copy()

    seen_keys = set(result[["junction_aa", "v_call", "j_call"]].itertuples(index=False, name=None))
    missing = target_n - len(result)
    print(f"Top-up: {path.name} -> need {missing} more unique rows")

    extra = read_unique_rows(
        path,
        chain,
        target_n=missing,
        skiprows=None,
        chunksize=chunksize,
        add_alleles=add_alleles,
        seen_keys=seen_keys,
    )

    result = pd.concat([result, extra], ignore_index=True)
    if len(result) < target_n:
        print(f"Warning: {path.name} -> only {len(result)} unique rows available in total")
    return result.head(target_n).copy()

made = []

for path in sorted(src_dir.glob("*_1e7.tsv.gz")):
    chain = path.name.split("_")[0].upper()
    if chain in exclude or chain not in targets:
        continue

    proto = read_unique_rows(path, chain, target_n=3000, add_alleles=True)
    proto.insert(0, "locus", chain)
    proto.insert(0, "clone_id", [f"{chain}_{i+1}" for i in range(len(proto))])

    out_path = dst_dir / f"{chain}_prototypes_3000.tsv"
    proto.to_csv(out_path, sep="\t", index=False)
    made.append(out_path.name)

print("Created:")
for name in made:
    print(" -", name)


Created:
 - IGH_prototypes_3000.tsv
 - IGK_prototypes_3000.tsv
 - IGL_prototypes_3000.tsv
 - TRD_prototypes_3000.tsv
 - TRG_prototypes_3000.tsv


In [8]:
from pathlib import Path
import pandas as pd

src_dir = Path("/projects/immunestatus/vdjrearm/sample")
dst_dir = Path("/projects/immunestatus/vdjrearm/airr_format")
dst_dir.mkdir(parents=True, exist_ok=True)

targets = {"TRG", "TRD", "IGH", "IGK", "IGL"}

airr_locus = {
    "TRG": "gamma",
    "TRD": "delta",
    "IGH": "heavy",
    "IGK": "kappa",
    "IGL": "lambda",
}

out_name = {
    "TRG": "trg_background_100k.tsv",
    "TRD": "trd_background_100k.tsv",
    "IGH": "igh_background_100k.tsv",
    "IGK": "igk_background_100k.tsv",
    "IGL": "igl_background_100k.tsv",
}

for path in sorted(src_dir.glob("*_1e7.tsv.gz")):
    chain = path.name.split("_")[0].upper()
    if chain not in targets:
        continue

    airr = read_unique_rows_with_topup(
        path,
        chain,
        target_n=100000,
        primary_skiprows=range(1, 10001),
        add_alleles=False,
    )

    airr["locus"] = airr_locus[chain]
    airr = airr[["junction_aa", "v_call", "j_call", "locus"]]

    airr.to_csv(dst_dir / out_name[chain], sep="\t", index=False)

print("Saved files:")
for chain in sorted(targets):
    p = dst_dir / out_name[chain]
    if p.exists():
        print(" -", p)


Saved files:
 - /projects/immunestatus/vdjrearm/airr_format/igh_background_100k.tsv
 - /projects/immunestatus/vdjrearm/airr_format/igk_background_100k.tsv
 - /projects/immunestatus/vdjrearm/airr_format/igl_background_100k.tsv
 - /projects/immunestatus/vdjrearm/airr_format/trd_background_100k.tsv
 - /projects/immunestatus/vdjrearm/airr_format/trg_background_100k.tsv
